# Phase 2 Price Prediction Baseline

Selected city: Stockholm, Sweden

This notebook is a readable companion to `src/run_phase2_ml.py`. The script is the reproducible source for the generated ML report and figures.

## Important Notes

- Target: `price` from `listings_clean`.
- Rows with missing listing price are excluded.
- Calendar prices are not used because they are 100% missing.
- This is a simple baseline, not a production model.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

project_root = Path.cwd()
db_path = project_root / 'data' / 'processed' / 'airbnb_stockholm.db'

features = [
    'room_type', 'property_type', 'neighbourhood_cleansed',
    'accommodates', 'bedrooms', 'bathrooms', 'beds',
    'minimum_nights', 'maximum_nights', 'availability_365',
    'number_of_reviews', 'review_scores_rating', 'host_is_superhost',
    'host_listings_count', 'instant_bookable',
]

with sqlite3.connect(db_path) as con:
    listings = pd.read_sql_query('SELECT price, ' + ', '.join(features) + ' FROM listings_clean', con)

listings.shape

In [ ]:
priced = listings.dropna(subset=['price']).copy()
price_cap_99 = priced['price'].quantile(0.99)
modeling_data = priced[priced['price'] <= price_cap_99].copy()

pd.Series({
    'original_rows': len(listings),
    'missing_price_rows': listings['price'].isna().sum(),
    'price_cap_99': price_cap_99,
    'modeling_rows': len(modeling_data),
})

## Generated Model Results

The full model training is implemented in `src/run_phase2_ml.py` and summarized in `reports/ml_experiment_report.md`.

![ML model comparison](../reports/figures/ml_model_comparison.png)

![ML feature importance](../reports/figures/ml_feature_importance.png)

## Reproducible Command

Run from the project root:

```bash
python src/run_phase2_ml.py
```

Generated report: `reports/ml_experiment_report.md`